# 07 · Sparkplug B

**Objetivo:** entender o que o Sparkplug B acrescenta ao MQTT: namespace fixo, **certificados de nascimento e morte**, **aliases**, **números de sequência** e payload **protobuf** tipado.

| MQTT puro deixa em aberto | Sparkplug B define |
|---|---|
| estrutura de tópicos | `spBv1.0/<grupo>/<tipo>/<nó>[/<dispositivo>]` |
| formato do payload | protobuf com tipo de dado por métrica |
| como saber o que existe | NBIRTH / DBIRTH listam todas as métricas |
| como saber que morreu | NDEATH via Last Will + `bdSeq` |
| como saber que perdeu algo | `seq` de 0 a 255 |

In [1]:
# Configuração: deixa a raiz do repositório no caminho de import
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
from common.notebook import load, run_in_background, require_broker
require_broker()
sp = load("07-sparkplug-b/sparkplug_demo.py")

broker MQTT encontrado em localhost:1883


## 1. Rodando o nó de borda e a aplicação host

O roteiro: o nó nasce (NBIRTH + DBIRTH), manda 6 DDATA, **perde uma mensagem de propósito**, manda mais uma, e morre sem desconectar.

In [2]:
resultado = sp.main()

Host application log:
  NBIRTH  edge-gw-01             seq=0    64 bytes  bdSeq, Node Control/Rebirth
  DBIRTH  edge-gw-01/filler-01   seq=1   176 bytes  Process/Speed, Process/Running, Counters/Good, Counters/Reject, Info/Recipe
  DDATA   edge-gw-01/filler-01   seq=2    57 bytes  Process/Speed, Process/Running, Counters/Good
  DDATA   edge-gw-01/filler-01   seq=3    57 bytes  Process/Speed, Process/Running, Counters/Good
  DDATA   edge-gw-01/filler-01   seq=4    57 bytes  Process/Speed, Process/Running, Counters/Good
  DDATA   edge-gw-01/filler-01   seq=5    57 bytes  Process/Speed, Process/Running, Counters/Good
  DDATA   edge-gw-01/filler-01   seq=6    57 bytes  Process/Speed, Process/Running, Counters/Good
  DDATA   edge-gw-01/filler-01   seq=7    57 bytes  Process/Speed, Process/Running, Counters/Good
  seq gap! expected 8, got 9 -> request rebirth
  DDATA   edge-gw-01/filler-01   seq=9    24 bytes  Counters/Reject
  NDEATH  edge-gw-01             seq=-    27 bytes  bdSeq

Current

**Interpretação do log**

1. **NBIRTH** (seq 0): o nó anuncia que está vivo, com `bdSeq` (número da sessão) e a métrica de controle `Rebirth`.
2. **DBIRTH** (seq 1): o dispositivo `filler-01` declara **todas as métricas**, com nome, tipo e **alias**. É o maior payload, porque leva os nomes.
3. **DDATA**: só os valores que mudaram, identificados pelo **alias numérico**. Bem menor.
4. **seq gap!**: pulamos um número de sequência de propósito; a aplicação host percebeu e, num sistema real, pediria um *rebirth* para se ressincronizar.
5. **NDEATH**: publicado **pelo broker** (era o Last Will), quando o socket caiu. O host marca o nó como offline.

## 2. Olhando um payload por dentro

In [3]:
import sparkplug_b_pb2 as spb
p = spb.Payload(seq=5)
sp.metric(p, alias=1, datatype=sp.FLOAT, value=417.5)
sp.metric(p, alias=3, datatype=sp.INT32, value=42)
bruto = p.SerializeToString()
print(f"{len(bruto)} bytes:", bruto.hex(" "))
print()
print(spb.Payload.FromString(bruto))

35 bytes: 12 10 10 01 18 96 bb c9 89 9d 34 20 09 65 00 c0 d0 43 12 0d 10 03 18 96 bb c9 89 9d 34 20 03 50 2a 18 05

metrics {
  alias: 1
  timestamp: 1794511101334
  datatype: 9
  float_value: 417.5
}
metrics {
  alias: 3
  timestamp: 1794511101334
  datatype: 3
  int_value: 42
}
seq: 5



In [4]:
import json
como_json = json.dumps({"Process/Speed": 417.5, "Counters/Good": 42, "timestamp": sp.now_ms(), "seq": 5})
print(f"protobuf com alias: {len(bruto)} bytes")
print(f"JSON equivalente  : {len(como_json)} bytes  ->  {como_json}")

protobuf com alias: 35 bytes
JSON equivalente  : 83 bytes  ->  {"Process/Speed": 417.5, "Counters/Good": 42, "timestamp": 1794511101339, "seq": 5}


**Interpretação:** o protobuf não manda nomes de campo, só números de campo e valores binários; o alias substitui o nome da métrica. Em links celulares pagos por megabyte, ou com milhares de tags, essa diferença é o motivo do Sparkplug existir.

## Resumo

* Sparkplug B = MQTT + **contrato**: qualquer host (Ignition, historiador, MES) entende qualquer nó sem configuração manual de tags.
* **Birth** dá o dicionário (nome, tipo, alias); **Data** manda só o que mudou; **Death** avisa a queda.
* `seq` e `bdSeq` permitem detectar perda e sessão trocada.
* Custo: payload binário (precisa do `.proto`) e um modelo mais rígido que o MQTT livre.